# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafiz-Taha-Hussain/Flyrank-Work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/Hafiz-Taha-Hussain/Flyrank-Work.git
%cd Flyrank-Work/work/notebooks

Cloning into 'Flyrank-Work'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 124 (delta 39), reused 86 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.84 MiB | 9.99 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Flyrank-Work/work/notebooks


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** Lane 2 (Refresh / Content Opportunity Scoring) doesn't end in a
single yes/no per page — it ends in an ordered queue a reviewer works down from the top.
That's the "which ones first?" shape from the task-type table, not "will this one decline"
in isolation. Underneath the ranking there's a classification-style signal (is this page a
declining, worth-reviewing candidate?), but the thing that actually gets consumed
downstream is the order, not the label — so I'm naming the outer task ranking/scoring, with
a proxy classification target feeding it (see section 2).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy: `is_declining_proxy`** = `trend_direction == "down"` AND `impressions_90d >= 100`
(declining somewhere that still has real search demand, not just noise).

**Where this label comes from — and why that's a problem I'm flagging, not hiding:**
this is a **defined rule**, not an observed outcome. `trend_direction` is itself computed
from `trend_pct`, comparing two windows inside the same 90-day snapshot — it isn't an
independent future event I waited to measure. Per the framing skill's own warning, training
directly on a rule-derived label risks the model just re-learning the rule instead of
learning something new about the world. I'm using it this week because it's the only label
available in the starter data (`is_declining_label` doesn't actually exist in this CSV — I
checked), but a genuinely observed target — e.g. clicks/impressions measured in a later,
currently-unseen window — is the honest fix to reach for once the warehouse data is
available.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df["is_declining_proxy"] = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
).astype(int)

print(df["is_declining_proxy"].value_counts())
print(f"\npositive rate: {df['is_declining_proxy'].mean()*100:.1f}%")

df.loc[
    df["is_declining_proxy"] == 1,
    ["content_id", "client_id", "trend_direction", "impressions_90d", "avg_position"]
].head(5)

is_declining_proxy
0    16848
1    13152
Name: count, dtype: int64

positive rate: 43.8%


,content_id,client_id,trend_direction,impressions_90d,avg_position
0,content_304f48230142,client_f369cb89fc,down,3803,10.6
1,content_a1fb4e703a9e,client_4e07408562,down,15320,20.3
2,content_9aa793d4d895,client_7f2253d7e2,down,12581,36.5
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140,44.0
5,content_d4084a4bc775,client_f369cb89fc,down,3970,8.5


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** A reviewer has roughly a week's worth of capacity, not the ability to
work through thousands of flagged pages — so the number that matters is: *of the top 50
pages the queue surfaces, how many are legitimate review candidates?* Plain accuracy would
be misleading here since the proxy is already ~44% positive (a model that just guesses
"declining" a lot would score deceptively well), and nobody downstream ever sees a raw
ROC-AUC — they see a ranked list and act on the top of it. This also lines up with the
starter pipeline's own reference numbers (baseline rule ≈ 0.240 precision@50, random forest
≈ 0.740 precision@50), which gives me something concrete to try to beat honestly in later
weeks.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (`content_id`), belonging to one pseudonymized `client_id`,
described by its trailing-90-day search and engagement signals.

In [5]:
print("rows, columns:", df.shape)
print("unique clients:", df["client_id"].nunique())

unit_cols = [
    "content_id", "client_id", "content_type", "word_count",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "engagement_rate", "days_since_last_update", "trend_direction",
    "is_declining_proxy",
]
df[unit_cols].head(10)

rows, columns: (30000, 45)
unique clients: 32


,content_id,client_id,content_type,word_count,impressions_90d,clicks_90d,avg_position,ctr,engagement_rate,days_since_last_update,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,29,10.6,0.76,5.88,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,7,20.3,0.05,0.00,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,11,36.5,0.09,0.00,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,58,6.2,0.49,1.28,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,24,44.0,0.13,0.00,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,3970,1,8.5,0.03,0.00,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,20,0,7.0,0.00,0.00,20,down,0
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,1724,1,21.2,0.06,3.57,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,3807.0,32574,29,46.0,0.09,5.88,20,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,NaN,1240,2,4.9,0.16,0.00,104,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The proxy rule alone already flags **13,152 of 30,000 rows (43.8%)** — and if I stack all
six of the starter baseline's individual reason-code rules together (stale visible,
declining with demand, thin visible, page-one decay, low CTR, low engagement), **21,024 of
30,000 rows (70.1%)** trip at least one of them. No reviewer works through 21,000 pages in
a week. A fixed rule can say "this page is a candidate," but it can't say *how much* it
matters relative to every other candidate — that depends on how several signals combine and
trade off for a specific page, and two pages can trip the same rule for very different
reasons and deserve very different priority. That weighting-and-combining problem is exactly
where a model earns its place over an if-statement: it can learn how much each signal should
count and how they interact, turning a flood of equally-weighted flags into an actual
ranking. The starter pipeline's own baseline-vs-model gap (0.240 vs. 0.740 precision@50) is
evidence this isn't just theoretical.

In [6]:
stale = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
thin = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
page_one_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
low_ctr = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
low_engagement = (df["sessions_90d"] >= 30) & ((df["engagement_rate"] < 30) | (df["scroll_rate"] < 30))

any_flag = stale | (df["is_declining_proxy"] == 1) | thin | page_one_decay | low_ctr | low_engagement
print(f"is_declining_proxy alone: {df['is_declining_proxy'].sum()} rows ({df['is_declining_proxy'].mean()*100:.1f}%)")
print(f"any of six baseline rules: {any_flag.sum()} rows ({any_flag.sum()/len(df)*100:.1f}%) — far past a ~50-page weekly capacity")

is_declining_proxy alone: 13152 rows (43.8%)
any of six baseline rules: 21024 rows (70.1%) — far past a ~50-page weekly capacity


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.